## Section 1 — Giới thiệu

Notebook này huấn luyện ARIMA/SARIMA trên 5 cặp (store_nbr, family) đại diện 5 pattern phát hiện qua EDA. Mục tiêu là thiết lập baseline cổ điển để so sánh RMSLE với LightGBM (Issue #3) trên cùng 5 series.

Kết quả lưu tại `arima_benchmark_results.csv` để dùng trong bước so sánh downstream.

In [1]:
import sys
from pathlib import Path

_here = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [_here] + list(_here.parents) if (p / 'config.py').exists()),
    _here
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import PROJECT_ROOT, DATA_DIR, RAW_DIR, PROCESSED_DIR, SPLITS_DIR

MODELS_DIR    = PROJECT_ROOT / 'models'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'


In [2]:
import warnings
warnings.filterwarnings('ignore')

import os
import time
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout
from statsmodels.tsa.statespace.sarimax import SARIMAX

BASE       = PROJECT_ROOT
TRAIN_PATH = PROCESSED_DIR / 'train_final.csv'
OUT_DIR    = NOTEBOOKS_DIR / '06_modeling' / 'ARIMA'
OUT_CSV    = OUT_DIR / 'arima_benchmark_results.csv'

TRAIN_END  = '2017-07-31'
VAL_START  = '2017-08-01'
VAL_END    = '2017-08-15'

print('Setup complete.')

Setup complete.


## Section 2 — Load dữ liệu

Load từ `data/processed/train_final.csv` — output của pipeline feature engineering (Issue #1). Áp dụng cùng ranh giới split: train <= 2017-04-27, val = 2017-04-28 -> 2017-05-12. Target được log1p transform nhất quán với Issue #1.

In [3]:
raw = pd.read_csv(TRAIN_PATH, parse_dates=['date'],
                  usecols=['date', 'store_nbr', 'family', 'sales'])
print(f"Loaded: {raw.shape} | {raw['date'].min().date()} -> {raw['date'].max().date()}")

Loaded: (3000888, 4) | 2013-01-01 -> 2017-08-15


In [4]:
train_raw = raw[raw['date'] <= TRAIN_END].copy()
val_raw   = raw[(raw['date'] >= VAL_START) & (raw['date'] <= VAL_END)].copy()

train_raw['sales_log'] = np.log1p(train_raw['sales'])
val_raw['sales_log']   = np.log1p(val_raw['sales'])

print(f"Train : {train_raw.shape} | {train_raw['date'].min().date()} -> {train_raw['date'].max().date()}")
print(f"Val   : {val_raw.shape}   | {val_raw['date'].min().date()} -> {val_raw['date'].max().date()}")

assert train_raw['date'].max() < val_raw['date'].min(), 'DATA LEAK'

sample = (
    train_raw[(train_raw['store_nbr'] == 1) & (train_raw['family'] == 'BEVERAGES')]
    [['date', 'sales', 'sales_log']].tail(3)
)
print('\nSample (store=1, BEVERAGES, last 3 train rows):')
print(sample.to_string(index=False))

Train : (2974158, 5) | 2013-01-01 -> 2017-07-31
Val   : (26730, 5)   | 2017-08-01 -> 2017-08-15



Sample (store=1, BEVERAGES, last 3 train rows):
      date  sales  sales_log
2017-07-29 2161.0   7.678789
2017-07-30 1212.0   7.100852
2017-07-31 2414.0   7.789455


## Section 3 — Chọn 5 series đại diện

Tiêu chí chọn 5 cặp (store_nbr, family) đại diện 5 pattern:
1. High & stable: mean cao, CV thấp
2. Low sales: mean thấp, zero_ratio < 30%
3. Clear seasonality: autocorrelation lag 7 cao nhất
4. Many zeros: zero_ratio > 30%, cao nhất
5. High volatility: CV cao nhất

In [5]:
grp = train_raw.groupby(['store_nbr', 'family'])['sales']

stats = pd.DataFrame({
    'mean'       : grp.mean(),
    'std'        : grp.std(),
    'max_val'    : grp.max(),
    'zero_ratio' : grp.apply(lambda x: (x == 0).mean()),
    'n_days'     : grp.count(),
}).reset_index()

stats['cv']        = stats['std'] / (stats['mean'] + 1e-9)
stats['max_ratio'] = stats['max_val'] / (stats['mean'] + 1e-9)

print(f'Total pairs: {len(stats)}')
print(stats[['mean', 'std', 'cv', 'zero_ratio']].describe().round(3).to_string())

Total pairs: 1782
           mean       std        cv  zero_ratio
count  1782.000  1782.000  1782.000    1782.000
mean    356.811   194.789     1.456       0.314
std     960.076   502.432     2.139       0.324
min       0.000     0.000     0.000       0.003
25%       2.842     3.550     0.459       0.003
50%      19.780    19.764     0.998       0.309
75%     220.749   132.020     1.462       0.509
max    9732.177  5584.676    28.899       1.000


In [6]:
cutoff_acf = pd.Timestamp('2016-04-28')
recent     = train_raw[train_raw['date'] >= cutoff_acf]

acf7_list = []
for (s, f), g in recent.groupby(['store_nbr', 'family']):
    series = g.sort_values('date')['sales'].values
    if len(series) > 14 and series.std() > 0:
        acf7 = float(pd.Series(series).autocorr(lag=7))
    else:
        acf7 = 0.0
    acf7_list.append({'store_nbr': s, 'family': f, 'acf_lag7': acf7})

acf_df = pd.DataFrame(acf7_list)
stats  = stats.merge(acf_df, on=['store_nbr', 'family'], how='left')
print(f'ACF lag-7 computed for {len(acf_df)} pairs.')

ACF lag-7 computed for 1782 pairs.


In [7]:
val_pairs = set(zip(val_raw['store_nbr'], val_raw['family']))
stats['in_val'] = stats.apply(
    lambda r: (r['store_nbr'], r['family']) in val_pairs, axis=1
)
sv = stats[stats['in_val']].copy()


def exclude_selected(df, sel):
    mask = df.apply(lambda r: (r['store_nbr'], r['family']) in sel, axis=1)
    return df[~mask]


chosen = []

# Pattern 1: high & stable
p1 = sv[sv['cv'] <= sv['cv'].quantile(0.50)].nlargest(1, 'mean').iloc[0]
chosen.append((p1['store_nbr'], p1['family']))

# Pattern 2: low sales (not zero-dominated)
p2 = exclude_selected(
    sv[(sv['zero_ratio'] < 0.30) & (sv['mean'] > 0)], chosen
).nsmallest(1, 'mean').iloc[0]
chosen.append((p2['store_nbr'], p2['family']))

# Pattern 3: clear seasonality
p3 = exclude_selected(sv, chosen).nlargest(1, 'acf_lag7').iloc[0]
chosen.append((p3['store_nbr'], p3['family']))

# Pattern 4: many zeros
p4 = exclude_selected(
    sv[sv['zero_ratio'] > 0.30], chosen
).nlargest(1, 'zero_ratio').iloc[0]
chosen.append((p4['store_nbr'], p4['family']))

# Pattern 5: high volatility
p5 = exclude_selected(sv, chosen).nlargest(1, 'cv').iloc[0]
chosen.append((p5['store_nbr'], p5['family']))

selection = [
    {'pattern': 'high_stable',
     'store_nbr': int(p1['store_nbr']), 'family': p1['family'],
     'justification': f"mean={p1['mean']:.1f}, cv={p1['cv']:.3f}"},
    {'pattern': 'low_sales',
     'store_nbr': int(p2['store_nbr']), 'family': p2['family'],
     'justification': f"mean={p2['mean']:.3f}, zero_ratio={p2['zero_ratio']:.3f}"},
    {'pattern': 'seasonality',
     'store_nbr': int(p3['store_nbr']), 'family': p3['family'],
     'justification': f"acf_lag7={p3['acf_lag7']:.3f}"},
    {'pattern': 'many_zeros',
     'store_nbr': int(p4['store_nbr']), 'family': p4['family'],
     'justification': f"zero_ratio={p4['zero_ratio']:.3f}"},
    {'pattern': 'high_volatility',
     'store_nbr': int(p5['store_nbr']), 'family': p5['family'],
     'justification': f"cv={p5['cv']:.3f}, max_ratio={p5['max_ratio']:.1f}"},
]

sel_df = pd.DataFrame(selection)
print('=== 5 Series Duoc Chon ===')
print(sel_df[['pattern', 'store_nbr', 'family', 'justification']].to_string(index=False))

=== 5 Series Duoc Chon ===
        pattern  store_nbr          family                justification
    high_stable         44       GROCERY I        mean=9732.2, cv=0.369
      low_sales         19          BEAUTY mean=1.590, zero_ratio=0.282
    seasonality         52           MEATS               acf_lag7=0.942
     many_zeros          1       BABY CARE             zero_ratio=1.000
high_volatility         19 LAWN AND GARDEN  cv=28.899, max_ratio=1112.7


## Section 4 — Huấn luyện mô hình và dự báo

Mô hình mặc định: SARIMAX(1,1,1)(1,1,1,7). Nếu fitting thất bại hoặc quá 60 giây, fallback về ARIMA(1,1,1). Dự báo 15 bước, inverse-transform bằng expm1, clip về 0 nếu âm. RMSLE tính thủ công.

In [8]:
def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))


def _fit_sarimax(series_log, order, seasonal_order):
    model = SARIMAX(series_log, order=order, seasonal_order=seasonal_order,
                    enforce_stationarity=False, enforce_invertibility=False)
    return model.fit(disp=False, maxiter=50, method='lbfgs')


def fit_with_timeout(series_log, order=(1, 1, 1), seasonal_order=(1, 1, 1, 7), timeout=60):
    with ThreadPoolExecutor(max_workers=1) as ex:
        fut = ex.submit(_fit_sarimax, series_log, order, seasonal_order)
        try:
            return fut.result(timeout=timeout), f'SARIMA{order}x{seasonal_order}'
        except (FuturesTimeout, Exception):
            return None, None


print('Helper functions defined.')

Helper functions defined.


In [9]:
results = {}

for row in selection:
    s       = row['store_nbr']
    f       = row['family']
    pattern = row['pattern']

    tr = train_raw[
        (train_raw['store_nbr'] == s) & (train_raw['family'] == f)
    ].sort_values('date')
    va = val_raw[
        (val_raw['store_nbr'] == s) & (val_raw['family'] == f)
    ].sort_values('date')

    train_log = tr['sales_log'].values
    val_true  = va['sales'].values
    n_train   = len(train_log)
    n_val     = len(val_true)

    t0 = time.time()
    fit_result, order_label = fit_with_timeout(
        train_log, order=(1, 1, 1), seasonal_order=(1, 1, 1, 7), timeout=60
    )

    if fit_result is None:
        print(f'  store {s} - {f}: SARIMA timeout/fail, fallback ARIMA(1,1,1)...')
        try:
            fit_result  = _fit_sarimax(train_log, (1, 1, 1), (0, 0, 0, 0))
            order_label = 'ARIMA(1,1,1)'
        except Exception as e:
            print(f'  store {s} - {f}: fallback failed ({e}). Skipping.')
            results[(s, f)] = {
                'pattern': pattern, 'rmsle': float('nan'),
                'n_train': n_train, 'n_val': n_val, 'model_order': 'FAILED'
            }
            continue

    elapsed   = time.time() - t0
    pred_log  = np.asarray(fit_result.get_forecast(steps=n_val).predicted_mean)
    pred_orig = np.clip(np.expm1(pred_log), 0, None)
    score     = rmsle(val_true, pred_orig)

    results[(s, f)] = {
        'pattern': pattern, 'rmsle': round(score, 4),
        'n_train': n_train, 'n_val': n_val, 'model_order': order_label
    }
    print(f'store {s} - {f}: {order_label}, RMSLE = {score:.4f}  ({elapsed:.0f}s)')

print('\nFitting complete.')

store 44 - GROCERY I: SARIMA(1, 1, 1)x(1, 1, 1, 7), RMSLE = 0.1951  (1s)


store 19 - BEAUTY: SARIMA(1, 1, 1)x(1, 1, 1, 7), RMSLE = 0.3556  (1s)


store 52 - MEATS: SARIMA(1, 1, 1)x(1, 1, 1, 7), RMSLE = 0.3098  (1s)


C:\Users\GIGABYTE\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


store 1 - BABY CARE: SARIMA(1, 1, 1)x(1, 1, 1, 7), RMSLE = 0.0000  (1s)


store 19 - LAWN AND GARDEN: SARIMA(1, 1, 1)x(1, 1, 1, 7), RMSLE = 0.0071  (1s)

Fitting complete.


In [10]:
import joblib
from sklearn.preprocessing import LabelEncoder

print('=== Evaluate XGBoost & LightGBM on exact same 15-day period ===')
SPLITS_V3 = PROCESSED_DIR / 'splits_safe_lag_v3'
X_train_ml = pd.read_csv(SPLITS_V3 / 'train_features.csv', usecols=['date', 'family', 'city', 'state', 'type', 'store_family'])
X_val_ml = pd.read_csv(SPLITS_V3 / 'val_features.csv', usecols=['date', 'family', 'city', 'state', 'type', 'store_family'])
X_test_ml = pd.read_csv(SPLITS_V3 / 'test_features.csv')

X_test_orig = X_test_ml.copy()

object_cols = X_test_ml.select_dtypes(include=['object']).columns.tolist()
for col in object_cols:
    le = LabelEncoder()
    combined = pd.concat([X_train_ml[col], X_val_ml[col], X_test_ml[col]], ignore_index=True).astype(str)
    le.fit(combined)
    X_test_ml[col] = le.transform(X_test_ml[col].astype(str)).astype(np.int32)

lgb_model = joblib.load(MODELS_DIR / 'lightgbm_safe_lag_v2_model.pkl')
xgb_model = joblib.load(MODELS_DIR / 'xgboost_safe_lag_v2_model.pkl')

X_test_orig['pred_lgb_log'] = np.expm1(lgb_model.predict(X_test_ml))
X_test_orig['pred_xgb_log'] = np.expm1(xgb_model.predict(X_test_ml))

ml_results = {}
for row in selection:
    s = row['store_nbr']
    f = row['family']
    mask = (X_test_orig['store_nbr'] == s) & (X_test_orig['family'] == f)
    sub = X_test_orig[mask].sort_values('date')
    
    # y_true is exactly val_true since we matched the dates (Aug 01-15)
    va = val_raw[(val_raw['store_nbr'] == s) & (val_raw['family'] == f)].sort_values('date')
    y_true = va['sales'].values
    
    score_lgb = rmsle(y_true, sub['pred_lgb_log'].values)
    score_xgb = rmsle(y_true, sub['pred_xgb_log'].values)
    ml_results[(s, f)] = {'lgb_rmsle': score_lgb, 'xgb_rmsle': score_xgb}
    print(f"store {s} - {f}: LGB_RMSLE = {score_lgb:.4f} | XGB_RMSLE = {score_xgb:.4f}")

=== Evaluate XGBoost & LightGBM on exact same 15-day period ===


store 44 - GROCERY I: LGB_RMSLE = 0.1465 | XGB_RMSLE = 0.1654
store 19 - BEAUTY: LGB_RMSLE = 0.3135 | XGB_RMSLE = 0.3164
store 52 - MEATS: LGB_RMSLE = 0.1243 | XGB_RMSLE = 0.1702
store 1 - BABY CARE: LGB_RMSLE = 0.0216 | XGB_RMSLE = 0.0330
store 19 - LAWN AND GARDEN: LGB_RMSLE = 0.0442 | XGB_RMSLE = 0.0435


## Section 5 — Kết quả và nhận xét

Bảng dưới là RMSLE của ARIMA/SARIMA trên 5 series, dùng làm baseline so sánh với LightGBM. Mô hình cổ điển được kỳ vọng kém hơn trên các series nhiều zero hoặc biến động cao.

In [11]:
rows = []
for (s, f), v in results.items():
    ml = ml_results.get((s, f), {'lgb_rmsle': float('nan'), 'xgb_rmsle': float('nan')})
    rows.append({
        'store_nbr'    : s,
        'family'       : f,
        'pattern'      : v['pattern'],
        'sarima_rmsle' : v['rmsle'],
        'lgb_rmsle'    : ml['lgb_rmsle'],
        'xgb_rmsle'    : ml['xgb_rmsle'],
        'n_train_days' : v['n_train'],
        'n_val_days'   : v['n_val'],
        'model_order'  : v['model_order'],
    })

result_df = pd.DataFrame(rows).sort_values('sarima_rmsle')

print('=== Benchmark Comparison: SARIMA vs LightGBM vs XGBoost ===')
print(result_df.to_string(index=False))

=== Benchmark Comparison: SARIMA vs LightGBM vs XGBoost ===
 store_nbr          family         pattern  sarima_rmsle  lgb_rmsle  xgb_rmsle  n_train_days  n_val_days                  model_order
         1       BABY CARE      many_zeros        0.0000   0.021593   0.032992          1669          15 SARIMA(1, 1, 1)x(1, 1, 1, 7)
        19 LAWN AND GARDEN high_volatility        0.0071   0.044188   0.043502          1669          15 SARIMA(1, 1, 1)x(1, 1, 1, 7)
        44       GROCERY I     high_stable        0.1951   0.146504   0.165388          1669          15 SARIMA(1, 1, 1)x(1, 1, 1, 7)
        52           MEATS     seasonality        0.3098   0.124332   0.170241          1669          15 SARIMA(1, 1, 1)x(1, 1, 1, 7)
        19          BEAUTY       low_sales        0.3556   0.313478   0.316399          1669          15 SARIMA(1, 1, 1)x(1, 1, 1, 7)


In [12]:
result_df.to_csv(OUT_CSV, index=False)
print(f'Saved: {OUT_CSV}')
print(f'Rows: {len(result_df)} | Columns: {list(result_df.columns)}')

Saved: D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series\notebooks\06_modeling\ARIMA\arima_benchmark_results.csv
Rows: 5 | Columns: ['store_nbr', 'family', 'pattern', 'sarima_rmsle', 'lgb_rmsle', 'xgb_rmsle', 'n_train_days', 'n_val_days', 'model_order']
